In [ ]:
# ============================================================
# Aether Oncology v3.0 — EDA: Oral Cancer Top 30 Countries
# Autor: Vitor Diogo Fonseca da Silva | FIAP Pós-Tech
# Dataset: Oral Cancer Prediction Dataset (160k records, MIT)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import LabelEncoder
import warnings
import logging

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("aether.eda")

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

logger.info("Setup concluído.")

In [ ]:
# ----------------------------------------------------------
# 1. CARREGAMENTO
# ----------------------------------------------------------
df = pd.read_csv("../data/raw/oral_cancer_top30.csv")

logger.info(f"Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

print("=== SHAPE ===")
print(df.shape)

print("\n=== PRIMEIRAS LINHAS ===")
display(df.head())

print("\n=== TIPOS E NULOS ===")
info = pd.DataFrame({
    "dtype": df.dtypes,
    "nulos": df.isnull().sum(),
    "nulos_%": (df.isnull().sum() / len(df) * 100).round(2),
    "únicos": df.nunique()
})
display(info)

In [ ]:
# ----------------------------------------------------------
# 2. DATA READINESS CHECKLIST
# ----------------------------------------------------------
print("=" * 50)
print("DATA READINESS CHECKLIST")
print("=" * 50)

checks = {
    "Volume mínimo (5.000+ registros)": len(df) >= 5000,
    "Features mínimas (10+)": df.shape[1] >= 10,
    "Sem coluna target ausente": "Diagnosis_Stage" in df.columns,
    "Nulos < 20% em qualquer coluna": (df.isnull().mean() < 0.2).all(),
    "Sem linhas 100% duplicadas": df.duplicated().sum() == 0,
}

for check, result in checks.items():
    status = "✅ PASS" if result else "❌ FAIL"
    print(f"  {status}  {check}")

print(f"\nTotal: {sum(checks.values())}/{len(checks)} checks passando")

In [ ]:
# ----------------------------------------------------------
# 3. TARGET BINÁRIO — JUSTIFICATIVA CLÍNICA
# ----------------------------------------------------------
# Early    → 0 (baixo risco, triagem padrão)
# Moderate → 1 (alto risco, intervenção prioritária)
# Late     → 1 (alto risco, intervenção urgente)

df["high_risk"] = df["Diagnosis_Stage"].isin(["Moderate", "Late"]).astype(int)

# Distribuição
counts = df["high_risk"].value_counts()
pct    = df["high_risk"].value_counts(normalize=True) * 100

print("=== DISTRIBUIÇÃO DO TARGET ===")
print(f"  Baixo risco (0 — Early):        {counts[0]:>7,} ({pct[0]:.1f}%)")
print(f"  Alto risco  (1 — Mod/Late):     {counts[1]:>7,} ({pct[1]:.1f}%)")
print(f"\n  Class imbalance ratio: {counts[0]/counts[1]:.2f}:1")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(
    [counts[0], counts[1]],
    labels=["Baixo Risco (Early)", "Alto Risco (Mod/Late)"],
    colors=["#4CAF50", "#F44336"],
    autopct="%1.1f%%",
    startangle=90
)
axes[0].set_title("Distribuição do Target Binário")

df["Diagnosis_Stage"].value_counts().plot(
    kind="bar", ax=axes[1],
    color=["#4CAF50", "#FF9800", "#F44336"],
    edgecolor="white"
)
axes[1].set_title("Distribuição por Estágio Original")
axes[1].set_xlabel("Estágio")
axes[1].set_ylabel("Contagem")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("../docs/eda_target_distribution.png", bbox_inches="tight")
plt.show()
logger.info("Target binário criado e visualizado.")

In [ ]:
# ----------------------------------------------------------
# 4. ANÁLISE DE FEATURES
# ----------------------------------------------------------

# 4.1 Distribuição de Idade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["Age"].hist(bins=30, ax=axes[0], color="#2196F3", edgecolor="white")
axes[0].set_title("Distribuição de Idade")
axes[0].set_xlabel("Idade")
axes[0].set_ylabel("Frequência")

sns.boxplot(data=df, x="high_risk", y="Age", ax=axes[1],
            palette={0: "#4CAF50", 1: "#F44336"})
axes[1].set_title("Idade por Nível de Risco")
axes[1].set_xticklabels(["Baixo Risco", "Alto Risco"])
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig("../docs/eda_age_distribution.png", bbox_inches="tight")
plt.show()

# 4.2 Features categóricas vs Target
cat_features = ["Gender", "Tobacco_Use", "Alcohol_Use", "Socioeconomic_Status"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    ct = pd.crosstab(df[feat], df["high_risk"], normalize="index") * 100
    ct.columns = ["Baixo Risco", "Alto Risco"]
    ct.plot(kind="bar", ax=axes[i], color=["#4CAF50", "#F44336"],
            edgecolor="white", rot=0)
    axes[i].set_title(f"{feat} vs Risco")
    axes[i].set_ylabel("% de casos")
    axes[i].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("../docs/eda_categorical_features.png", bbox_inches="tight")
plt.show()

# 4.3 Top 15 países por taxa de alto risco
country_risk = (
    df.groupby("Country")["high_risk"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    * 100
)

plt.figure(figsize=(12, 5))
country_risk.plot(kind="bar", color="#E91E63", edgecolor="white")
plt.title("Top 15 Países — Taxa de Alto Risco (%)")
plt.ylabel("% Alto Risco")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../docs/eda_country_risk.png", bbox_inches="tight")
plt.show()

logger.info("Análise de features concluída.")

In [ ]:
# ----------------------------------------------------------
# 5. CORRELAÇÃO (features numéricas + encoded)
# ----------------------------------------------------------
df_encoded = df.copy()

# Encode binárias para correlação
for col in ["Gender", "Tobacco_Use", "Alcohol_Use"]:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

# Encode ordinal do estágio
stage_map = {"Early": 0, "Moderate": 1, "Late": 2}
df_encoded["Diagnosis_Stage_num"] = df_encoded["Diagnosis_Stage"].map(stage_map)

num_cols = ["Age", "Gender", "Tobacco_Use", "Alcohol_Use",
            "Survival_Rate", "Diagnosis_Stage_num", "high_risk"]

corr = df_encoded[num_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, linewidths=0.5,
    annot_kws={"size": 10}
)
plt.title("Matriz de Correlação — Aether Oral Cancer")
plt.tight_layout()
plt.savefig("../docs/eda_correlation_heatmap.png", bbox_inches="tight")
plt.show()

logger.info("Correlação calculada.")

In [ ]:
# ----------------------------------------------------------
# 6. CALIBRATION PLOT (Reliability Diagram)
# Demonstra que as probabilidades do modelo são confiáveis
# ----------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Pipeline simples para gerar probabilidades de referência
X = df[["Age", "Gender", "Tobacco_Use", "Alcohol_Use", "Socioeconomic_Status"]].copy()
y = df["high_risk"]

preprocessor = ColumnTransformer([
    ("num", "passthrough", ["Age"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     ["Gender", "Tobacco_Use", "Alcohol_Use", "Socioeconomic_Status"]),
])

pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(random_state=SEED))])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

pipe.fit(X_train, y_train)
prob_pos = pipe.predict_proba(X_test)[:, 1]

# Calibration curve
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, prob_pos, n_bins=10
)

plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], "k--", label="Perfeitamente Calibrado")
plt.plot(mean_predicted_value, fraction_of_positives,
         "s-", color="#E91E63", linewidth=2, markersize=8,
         label="Modelo Aether (Baseline LR)")

plt.fill_between(mean_predicted_value, fraction_of_positives,
                 mean_predicted_value, alpha=0.1, color="#E91E63")

plt.xlabel("Probabilidade Predita Média", fontsize=12)
plt.ylabel("Fração de Positivos Reais", fontsize=12)
plt.title("Calibration Plot (Reliability Diagram)\nAether Oncology — Oral Cancer", fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/eda_calibration_plot.png", bbox_inches="tight")
plt.show()

print("Quanto mais próximo da diagonal tracejada, melhor a calibração.")
logger.info("Calibration plot gerado — docs/eda_calibration_plot.png")

In [ ]:
# ----------------------------------------------------------
# 7. ANÁLISE DE FAIRNESS POR SUBGRUPOS
# Antecipa o Fairlearn do v3.0 com evidência estática
# ----------------------------------------------------------
print("=" * 55)
print("ANÁLISE DE FAIRNESS — DISTRIBUIÇÃO DE RISCO POR GRUPO")
print("=" * 55)

# Por gênero
print("\n📊 Por Gênero:")
display(
    df.groupby("Gender")["high_risk"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Taxa Alto Risco", "count": "N"})
    .assign(**{"Taxa Alto Risco": lambda x: (x["Taxa Alto Risco"] * 100).round(1)})
)

# Por uso de tabaco
print("\n📊 Por Uso de Tabaco:")
display(
    df.groupby("Tobacco_Use")["high_risk"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Taxa Alto Risco", "count": "N"})
    .assign(**{"Taxa Alto Risco": lambda x: (x["Taxa Alto Risco"] * 100).round(1)})
)

# Por status socioeconômico
print("\n📊 Por Status Socioeconômico:")
display(
    df.groupby("Socioeconomic_Status")["high_risk"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Taxa Alto Risco", "count": "N"})
    .assign(**{"Taxa Alto Risco": lambda x: (x["Taxa Alto Risco"] * 100).round(1)})
    .sort_values("Taxa Alto Risco", ascending=False)
)

# Plot fairness
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title in zip(axes,
    ["Gender", "Tobacco_Use", "Socioeconomic_Status"],
    ["Gênero", "Tabaco", "Status Socioeconômico"]):

    rates = df.groupby(col)["high_risk"].mean() * 100
    rates.plot(kind="barh", ax=ax, color="#9C27B0", edgecolor="white")
    ax.set_title(f"Alto Risco por {title}")
    ax.set_xlabel("% Alto Risco")
    ax.axvline(df["high_risk"].mean() * 100, color="red",
               linestyle="--", alpha=0.7, label="Média geral")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../docs/eda_fairness_subgroups.png", bbox_inches="tight")
plt.show()

logger.info("Análise de fairness concluída — docs/eda_fairness_subgroups.png")